# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anjds22/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Signal 1 — GSC impressions: CONFIRMED. Low-impression content is common in the March 2026 data, so impressions provide a useful search-visibility signal for prioritising content.
Signal 2 — GSC average position: CONFIRMED. A substantial number of content observations have an average position of 20 or worse, so search position provides a useful signal for identifying content that may need attention.
Rule: Prioritise content with low search impressions and poor average search position.
Reason code: LOW_VISIBILITY
Action: REVIEW
The score will give higher priority to content with fewer impressions and a worse average search position.

In [18]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

In [19]:
REL = "hf://datasets/FlyRank/internship-warehouse"

fact_daily = f"""
read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
"""

In [ ]:
# Build the baseline inputs from March 2026 only

baseline = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_avg_position
FROM {fact_daily}
WHERE month = '2026-03'
  AND gsc_data_available = TRUE
""").df()

print("Rows:", len(baseline))
baseline.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import os
os.makedirs("work/outputs", exist_ok=True)

In [ ]:
# Build a simple baseline score
# Higher score = higher priority for review

baseline["score"] = (
    (baseline["gsc_impressions"] < 100).astype(int) +
    (baseline["gsc_avg_position"] >= 20).astype(int)
)

baseline["reason_code"] = "LOW_VISIBILITY"
baseline["action"] = "REVIEW"

# Rank highest-priority rows first
baseline = baseline.sort_values(
    ["score", "gsc_impressions", "gsc_avg_position"],
    ascending=[False, True, False]
).reset_index(drop=True)

baseline["rank"] = baseline.index + 1

# Save the ranked queue
output_path = "work/outputs/baseline_action_score.csv"

baseline[
    [
        "rank",
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "score",
        "reason_code",
        "action",
        "gsc_impressions",
        "gsc_avg_position"
    ]
].to_csv(output_path, index=False)

print(f"Saved {len(baseline)} rows to {output_path}")
baseline.head(10)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top 20 observations are all assigned the REVIEW action with the LOW_VISIBILITY reason code because they have both very low GSC impressions and poor average search position. Confidence is moderate because these two signals identify low search visibility but do not establish the cause. The rule could be wrong if the content is intentionally low-volume, newly published, highly specialised, or if the low visibility is caused by factors that cannot be observed in this dataset.

In [ ]:
# Top-20 review table

top20 = baseline.head(20).copy()

top20["confidence_note"] = (
    "Moderate: low visibility signals are present, "
    "but the rule does not identify the cause."
)

top20["what_would_make_it_wrong"] = (
    "Could be wrong if the content is intentionally low-volume, "
    "new, specialised, or affected by an unobserved factor."
)

top20[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The weakest picks are observations with very low impressions that may not necessarily need action. Low visibility could be intentional, caused by a newly published page, a specialised topic, or another factor not represented in the dataset. The baseline therefore identifies candidates for review rather than proving that a content change is required.
The baseline uses only March 2026 observations and the contemporaneous fields gsc_impressions and gsc_avg_position. It does not use future-window information, future performance labels, or product flags.

In [ ]:
# Weak picks and leakage check

print("WEAKEST PICKS:")
print(
    baseline.tail(10)[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "score",
            "gsc_impressions",
            "gsc_avg_position"
        ]
    ].to_string(index=False)
)

print("\nLEAKAGE CHECK:")

print(
    "Uses only March 2026:",
    baseline["report_date"].dt.to_period("M").eq("2026-03").all()
)

print("Uses future-window fields: NO")
print("Uses future labels: NO")
print("Uses product flags: NO")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.